# <font color="#003660">Applied Machine Learning for Text Analysis (M.184.5331)</font>

# <font color="#003660">Session 6: Structured Outputs</font>

# <font color="#003660">XML Outputs with Verifiers</font>

<center><br><img width=256 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/dag.png"/><br></center>

<p>

<div>
    <font color="#085986"><b>By the end of this lesson, you ...</b><br><br>
        ... will know how to generate XML and check it with `ElementTree`, <br>
        ... will know how to use the verifier and the retry from 6.01 for XML, <br>
        ... will know what a code verifier cannot check.
    </font>
</div>
</p>

Inspired by these excellent sources:

* [LangChain Docs: XMLOutputParser](https://python.langchain.com/docs/how_to/output_parser_xml/)
* [Python Docs: xml.etree.ElementTree](https://docs.python.org/3/library/xml.etree.elementtree.html)

In [ ]:
%pip install -q langchain langchain-openai python-dotenv tqdm

## Setup

The same setup as in 6.01: you choose where the model runs, and LangChain talks to it with `ChatOpenAI`.

| `provider` | Where the model runs | `model_name` | Good to know |
|---|---|---|---|
| `"ollama"` | **Ollama** on your computer | `qwen3.5:9b`, `qwen3.5:4b`, `qwen3.5:2b` | free, no account; install [Ollama](https://ollama.com/download) first, then download the model once in a terminal: `ollama pull qwen3.5:9b` (6.6 GB; 4b: 3.3 GB; 2b: 2.7 GB); without a GPU, it runs on the CPU, which is slower |
| `"colab"` | **Ollama on Google Colab** | the same three | free, needs a Google account; the cell installs Ollama in the Colab runtime and downloads the model, as in 4.01; choose a GPU runtime first (Runtime > Change runtime type > T4 GPU) |
| `"huggingface"` | **Hugging Face endpoints** of the course | the same three, one endpoint each | key and addresses from the course; if an endpoint was asleep, the first request takes 1 to 2 minutes |
| `"ai-gateway"` | **AI-Gateway** of Paderborn University | `gwdg.qwen3.8-27b-no-think` (thinking off), `gwdg.qwen3.8-27b` (thinking on) | your university key (see [setup/SETUP_AI_GATEWAY.md](https://github.com/olivermueller/amlta-2026/blob/main/setup/SETUP_AI_GATEWAY.md)); only in the university network or the VPN, not on Colab; about 10 requests per minute |
| `"other"` | **another** OpenAI-compatible API | the name your provider uses | base URL and key in `.env` (`OTHER_BASE`, `OTHER_KEY`) |

Keys and addresses come from the `.env` file in the repo folder (see `.env.example`). Without a `.env` file, for example on Colab, the cell asks for them.

Some models **think** before they answer (4.02, Part 3), and the thinking counts towards `max_tokens`. Every `ChatOpenAI` sets it in its `extra_body`: `"reasoning_effort": "none"` switches it off (Ollama, Hugging Face endpoint), `"medium"` switches it on. On the AI-Gateway, the model name decides (`-no-think`).

In [ ]:
# Run this cell: it only defines setup_model(). We call it before every ChatOpenAI, with the provider and the model.
import getpass
import json
import os
import shutil
import subprocess
import sys
import time
import urllib.error
import urllib.request

from dotenv import load_dotenv

load_dotenv()  # reads the .env file if there is one: this folder first, then the folders above (the repo folder)
IN_COLAB = "google.colab" in sys.modules

PROVIDERS = ["ollama", "colab", "huggingface", "ai-gateway", "other"]
QWEN = {  # model_name: (download size in Ollama, the name of its endpoint address in .env, the model on Hugging Face)
    "qwen3.5:9b": ("6.6 GB", "HF_GATEWAY_BASE_9B", "unsloth/Qwen3.5-9B-GGUF"),
    "qwen3.5:4b": ("3.3 GB", "HF_GATEWAY_BASE_4B", "unsloth/Qwen3.5-4B-GGUF"),
    "qwen3.5:2b": ("2.7 GB", "HF_GATEWAY_BASE_2B", "unsloth/Qwen3.5-2B-GGUF"),
}
GATEWAY_QWEN = {  # on the AI-Gateway, the model name decides about thinking
    "gwdg.qwen3.8-27b-no-think": "thinking off",
    "gwdg.qwen3.8-27b": "thinking on",
}


def value(env_name, question, secret=False):
    """Takes the value from .env, or asks for it (keys are typed hidden)."""
    found = os.getenv(env_name)
    if found:
        return found.strip()
    answer = getpass.getpass(question).strip() if secret else input(question).strip()
    os.environ[env_name] = answer  # asked once: the next call of setup_model() takes it from here
    return answer


def start_ollama(model, size, colab=False):
    """Makes sure that the Ollama server runs and has the model. Downloads the model only on Colab."""
    url = (os.getenv("OLLAMA_BASE") or "http://localhost:11434").rstrip("/")

    def running():
        try:
            urllib.request.urlopen(url, timeout=2)
            return True
        except OSError:
            return False

    if colab and shutil.which("ollama") is None:
        # the same steps as in 4.01: the installer needs pciutils and zstd
        print("Installing Ollama in the Colab runtime (about a minute) ...")
        subprocess.run("sudo apt-get install -y -qq pciutils zstd > /dev/null && curl -fsSL https://ollama.com/install.sh | sh > /dev/null",
                       shell=True, check=True)
    if not running():
        if shutil.which("ollama") is None:
            raise RuntimeError("Ollama is not installed. Install it from https://ollama.com/download and run this cell again.")
        # on Colab, the installer may say that the server is running, but it is not; so we start it ourselves (as in 4.01)
        print("Starting the Ollama server ...")
        ollama_log = open("ollama.log", "w")  # a pipe that nobody reads could fill up and block the server
        subprocess.Popen(["ollama", "serve"], stdout=ollama_log, stderr=ollama_log)
        for _ in range(30):
            if running():
                break
            time.sleep(1)
        else:
            raise RuntimeError("The Ollama server did not start. Look into ollama.log.")

    installed = [m["name"] for m in json.load(urllib.request.urlopen(url + "/api/tags"))["models"]]
    if model not in installed and not colab:  # on your computer, you download the model yourself, once
        raise RuntimeError(f"{model} is not downloaded yet. Run this command in a terminal (it downloads {size}), "
                           f"then run this cell again:\n\n    ollama pull {model}\n")
    if model not in installed:
        print(f"WARNING: {model} is not here yet. It is downloaded now ({size}), this can take several minutes.")
        request = urllib.request.Request(url + "/api/pull", data=json.dumps({"model": model}).encode())
        shown = -1
        for line in urllib.request.urlopen(request):
            status = json.loads(line)
            if "error" in status:
                raise RuntimeError(status["error"])
            if status.get("total"):
                percent = int(100 * status.get("completed", 0) / status["total"]) // 10 * 10
                if percent != shown:
                    print(f"  {percent} %", end="  ")
                    shown = percent
        print("\nDownload finished.")
    return url + "/v1/"


def wait_until_awake(base_url, api_key):
    """A sleeping Hugging Face endpoint answers 503 until it has started; we wait for it."""
    request = urllib.request.Request(base_url + "/models", headers={"Authorization": f"Bearer {api_key}"})
    for attempt in range(40):  # up to about 6 minutes
        try:
            urllib.request.urlopen(request, timeout=10)
            return
        except urllib.error.HTTPError as error:
            if error.code == 403:
                raise RuntimeError("The endpoint answers 403: it is paused (the course endpoints only run during the seminar), "
                                   "or your key has no access to it. Choose another provider, or check HF_GATEWAY_KEY and the address in .env.")
            if error.code != 503:
                raise RuntimeError(f"The endpoint answers {error.code}. Check the key and the address.")
        except OSError:
            pass
        if attempt == 0:
            print("The endpoint is asleep and starts now. This takes a few minutes ...")
        time.sleep(8)
    raise RuntimeError("The endpoint did not wake up. Try again in a few minutes.")


def setup_model(model):
    """Checks {"model_name": ..., "provider": ...}, sets the address and the key, and prints what to keep in mind.
    Returns base_url, api_key and the model name for ChatOpenAI."""
    name = model.get("model_name", "").strip()
    provider = model.get("provider", "").strip().lower()
    if provider not in PROVIDERS:
        raise ValueError(f"Unknown provider {provider!r}. Choose one of: {', '.join(PROVIDERS)}.")
    if provider in ("ollama", "colab", "huggingface") and name not in QWEN:
        raise ValueError(f"The provider {provider!r} has these models: {', '.join(QWEN)}. Not {name!r}.")
    notes = []

    if provider in ("ollama", "colab"):
        if provider == "ollama" and IN_COLAB:
            raise ValueError("This notebook runs on Google Colab: use the provider 'colab'.")
        if provider == "colab" and not IN_COLAB:
            raise ValueError("The provider 'colab' only works on Google Colab. On your own computer, use 'ollama'.")
        if provider == "colab" and subprocess.run("nvidia-smi", shell=True, capture_output=True).returncode != 0:
            notes.append("This Colab runtime has no GPU: answers take long. Better: Runtime > Change runtime type > T4 GPU, then run the notebook again.")
        if provider == "ollama":
            notes.append("The model runs on this computer. Without a GPU, it runs on the CPU, and answers take longer.")
        base_url, api_key = start_ollama(name, QWEN[name][0], colab=(provider == "colab")), "ollama"  # Ollama needs no key
        notes.append('Thinking: switch it with "reasoning_effort" in the extra_body of ChatOpenAI ("none" = off).')
        notes.append("Several answers per request (n > 1): Ollama returns only one.")

    elif provider == "huggingface":
        _, env_name, name = QWEN[name]  # name: the model on Hugging Face
        base_url = value(env_name, f"Paste the address of the Hugging Face endpoint for {model['model_name']}: ").rstrip("/")
        if "<" in base_url:
            raise ValueError(f"The address for {model['model_name']} is still a placeholder in .env ({env_name}). "
                             "Paste the address from the course there, or choose another model or provider.")
        if not base_url.endswith("/v1"):
            base_url += "/v1"  # the OpenAI-compatible API of the endpoint
        api_key = value("HF_GATEWAY_KEY", "Paste your Hugging Face key (for a public endpoint, just press Enter): ", secret=True)
        api_key = api_key or "none"  # a public endpoint needs no key, but the client needs some text
        wait_until_awake(base_url, api_key)
        notes.append('Thinking: switch it with "reasoning_effort" in the extra_body of ChatOpenAI ("none" = off).')
        notes.append("Several answers per request (n > 1): the endpoint refuses them (HTTP 400).")

    elif provider == "ai-gateway":
        if IN_COLAB:
            notes.append("Colab runs outside the university network and usually cannot reach the AI-Gateway.")
        base_url = os.getenv("AI_GATEWAY_BASE") or "https://ai-gateway.uni-paderborn.de/v1"
        api_key = value("AI_GATEWAY_KEY", "Paste your AI-Gateway key: ", secret=True)
        if name not in GATEWAY_QWEN:
            notes.append(f"{name!r} is not one of the Qwen models of the course ({', '.join(GATEWAY_QWEN)}). "
                         "llm.root_client.models.list() lists the models of your account.")
        notes.append("Thinking cannot be switched with reasoning_effort here (the gateway ignores it): the model name decides. "
                     "gwdg.qwen3.8-27b-no-think = off, gwdg.qwen3.8-27b = on.")
        notes.append("Only in the university network or the VPN. About 10 requests per minute: if a cell pauses, the client waits and continues.")

    else:  # another OpenAI-compatible API
        if not name:
            raise ValueError("Give the model name of your provider as model_name.")
        base_url = value("OTHER_BASE", "Base URL of the API (e.g. https://api.openai.com/v1/): ")
        api_key = value("OTHER_KEY", "Your API key: ", secret=True)
        notes.append("Check in the documentation of your provider how thinking is switched; "
                     "the extra_body of ChatOpenAI may need other fields than reasoning_effort.")

    print(f"Provider: {provider} | model: {name}")  # the key is never printed
    print("Keep in mind:")
    for note in notes:
        print("  -", note)
    return base_url, api_key, name

**Before every `ChatOpenAI`, we call `setup_model()`** with the provider and the model, for example `setup_model({"model_name": "qwen3.5:9b", "provider": "huggingface"})`. It checks whether the combination exists, sets the address and the key, and prints what to keep in mind. It returns three values, which we split directly into `BASE_URL`, `API_KEY`, and `MODEL_NAME`. To switch the model, change the dict there.

In [ ]:
# packages
import os
import xml.etree.ElementTree as ET
from typing import Optional

from tqdm.notebook import tqdm
from langchain_openai import ChatOpenAI
from langchain_core.messages import convert_to_messages
from langchain_core.output_parsers import XMLOutputParser

To use a model with LangChain, we choose it with `setup_model()` and create a `ChatOpenAI` instance, as in 6.01.

In [ ]:
# Initialize the OpenAI LLM with global configuration
BASE_URL, API_KEY, MODEL_NAME = setup_model(
    {
        "model_name": "qwen3.5:2b", 
        "provider": "huggingface"
        }
)  # provider and model: see the table in the setup
llm = ChatOpenAI(
    model=MODEL_NAME,  # on the AI-Gateway, the model decides about thinking: gwdg.qwen3.8-27b-no-think (off) or gwdg.qwen3.8-27b (on)
    base_url=BASE_URL,
    api_key=API_KEY,
    # seed=42, # intentionally not setting the seed to get different answers each time 
    temperature=1.0,
    extra_body={
        "max_tokens": 1024,  # the token limit (here, because Ollama ignores max_completion_tokens, the name that ChatOpenAI uses)
        "reasoning_effort": "none",  # thinking off; "low", "medium", or "high" switches it on (Ollama, Hugging Face endpoint)
    },
)

## The Business Case

We stay with the online shop from 6.01: the same catalog and the first eight support e-mails (T-2005 now has its order number). New is the shop's database `shop.db` with the customers and their orders. The verifier needs it for the business rules. The cells are ready: just run them.

In [ ]:
# the product catalog of our shop: SKU -> (name, price in EUR)
CATALOG = {
    "KB-100": ("Wireless Keyboard", 49.90),
    "MS-200": ("Ergonomic Mouse", 29.90),
    "MN-270": ("27-inch Monitor", 249.00),
    "HD-300": ("Noise-Cancelling Headset", 129.00),
    "DK-400": ("USB-C Docking Station", 159.00),
    "LP-500": ("Laptop Stand", 39.90),
    "WC-600": ("HD Webcam", 79.00),
}

# eight support e-mails from the last weeks
TICKETS = {
    "T-2001": """Hi, the docking station I received last week does not charge my laptop at all. I tried two different cables.
I have a client presentation on Monday and cannot work like this. Please send a replacement as fast as possible.
Anna Schmidt, order ORD-10001""",
    "T-2002": """Hello,
I ordered two 27 inch monitors, but you charged my credit card twice for the same order (2 x 498 EUR).
Please refund the duplicate payment. Order number: ORD-10002
Regards, Jonas Weber""",
    "T-2003": """Where is my headset? The tracking has not changed for 5 days (order ORD-10003).
No rush, I just want to know when it arrives. Lena Fischer""",
    "T-2004": """We ordered 3 ergonomic mice and 3 laptop stands for our team. One of the laptop stands arrived broken,
the other two and the mice are fine. Could you replace the broken one? We can use the old stands until then.
- Mehmet Yilmaz, ORD-10004""",
    "T-2005": """I want to cancel my order for the webcam, I found it cheaper somewhere else.
It is order ORD-10005 from September 20. Please cancel it and refund the money. Sofia Rossi""",
    "T-2006": """Hi team, I would like to return the two wireless keyboards from order ORD-10007, they do not fit my desk.
They are unused and still in the box. I keep the mice. Can I get my money back?
Tom Becker""",
    "T-2007": """URGENT!!! My order ORD-10008 with 2 docking stations was supposed to arrive yesterday,
for our new employees who start TODAY. Where are they??? If they do not arrive by tomorrow, I will order somewhere else.
Clara Wagner""",
    "T-2008": """Hello, I need an invoice with my company name (Klein Consulting GmbH) for my last order ORD-10009
(monitor and laptop stand). The current invoice only shows my private name.
Thanks, David Klein""",
}

In [ ]:
# the correct answers, checked by a person (the "gold standard")
GOLD = {
    "T-2001": {"customer_name": "Anna Schmidt", "order_id": "ORD-10001", "category": "defect", "priority": "high",
               "items": [{"sku": "DK-400", "quantity": 1}], "requested_action": "replacement"},
    "T-2002": {"customer_name": "Jonas Weber", "order_id": "ORD-10002", "category": "billing", "priority": "high",
               "items": [{"sku": "MN-270", "quantity": 2}], "requested_action": "refund"},
    "T-2003": {"customer_name": "Lena Fischer", "order_id": "ORD-10003", "category": "delivery", "priority": "low",
               "items": [{"sku": "HD-300", "quantity": 1}], "requested_action": "information"},
    "T-2004": {"customer_name": "Mehmet Yilmaz", "order_id": "ORD-10004", "category": "defect", "priority": "medium",
               "items": [{"sku": "LP-500", "quantity": 1}], "requested_action": "replacement"},
    "T-2005": {"customer_name": "Sofia Rossi", "order_id": "ORD-10005", "category": "cancellation", "priority": "low",
               "items": [{"sku": "WC-600", "quantity": 1}], "requested_action": "refund"},
    "T-2006": {"customer_name": "Tom Becker", "order_id": "ORD-10007", "category": "return", "priority": "low",
               "items": [{"sku": "KB-100", "quantity": 2}], "requested_action": "refund"},
    "T-2007": {"customer_name": "Clara Wagner", "order_id": "ORD-10008", "category": "delivery", "priority": "high",
               "items": [{"sku": "DK-400", "quantity": 2}], "requested_action": "information"},
    "T-2008": {"customer_name": "David Klein", "order_id": "ORD-10009", "category": "billing", "priority": "low",
               "items": [{"sku": "MN-270", "quantity": 1}, {"sku": "LP-500", "quantity": 1}], "requested_action": "invoice_correction"},
}

In [ ]:
import sqlite3

CUSTOMERS = [  # customer_id, name, city, customer since
    (1, "Anna Schmidt", "Paderborn", "2024-03-12"), (2, "Jonas Weber", "Bielefeld", "2025-01-08"),
    (3, "Lena Fischer", "Dortmund", "2025-06-30"), (4, "Mehmet Yilmaz", "Paderborn", "2023-11-02"),
    (5, "Sofia Rossi", "Köln", "2026-02-14"), (6, "Tom Becker", "Münster", "2024-09-21"),
    (7, "Clara Wagner", "Paderborn", "2022-05-17"), (8, "David Klein", "Bielefeld", "2026-07-01"),
]
ORDERS = [  # order_id, customer_id, order date, status
    ("ORD-10001", 1, "2026-09-02", "delivered"), ("ORD-10002", 2, "2026-09-05", "delivered"),
    ("ORD-10003", 3, "2026-09-10", "shipped"), ("ORD-10004", 4, "2026-09-12", "delivered"),
    ("ORD-10005", 5, "2026-09-20", "cancelled"), ("ORD-10006", 1, "2026-10-01", "delivered"),
    ("ORD-10007", 6, "2026-10-03", "delivered"), ("ORD-10008", 7, "2026-10-08", "shipped"),
    ("ORD-10009", 8, "2026-10-15", "delivered"), ("ORD-10010", 2, "2026-10-20", "delivered"),
]
ORDER_ITEMS = [  # order_id, sku, quantity
    ("ORD-10001", "DK-400", 1), ("ORD-10001", "KB-100", 1), ("ORD-10002", "MN-270", 2), ("ORD-10003", "HD-300", 1),
    ("ORD-10004", "MS-200", 3), ("ORD-10004", "LP-500", 3), ("ORD-10005", "WC-600", 1), ("ORD-10006", "WC-600", 1),
    ("ORD-10007", "KB-100", 2), ("ORD-10007", "MS-200", 2), ("ORD-10008", "DK-400", 2), ("ORD-10009", "MN-270", 1),
    ("ORD-10009", "LP-500", 1), ("ORD-10010", "HD-300", 2),
]
TICKET_ROWS = [  # ticket_id, order_id, customer_id, created, category, priority, status
    ("T-2001", "ORD-10001", 1, "2026-09-09", "defect", "high", "open"),
    ("T-2002", "ORD-10002", 2, "2026-09-07", "billing", "high", "closed"),
    ("T-2003", "ORD-10003", 3, "2026-09-18", "delivery", "low", "open"),
    ("T-2004", "ORD-10004", 4, "2026-09-16", "defect", "medium", "open"),
    ("T-2005", "ORD-10005", 5, "2026-09-21", "cancellation", "low", "closed"),
    ("T-2006", "ORD-10007", 6, "2026-10-09", "return", "low", "open"),
    ("T-2007", "ORD-10008", 7, "2026-10-13", "delivery", "high", "open"),
    ("T-2008", "ORD-10009", 8, "2026-10-20", "billing", "low", "closed"),
]

if os.path.exists("shop.db"):
    os.remove("shop.db")  # start fresh every time
with sqlite3.connect("shop.db") as con:
    con.executescript("""
        CREATE TABLE products (sku TEXT PRIMARY KEY, name TEXT, price REAL);
        CREATE TABLE customers (customer_id INTEGER PRIMARY KEY, name TEXT, city TEXT, customer_since TEXT);
        CREATE TABLE orders (order_id TEXT PRIMARY KEY, customer_id INTEGER REFERENCES customers, order_date TEXT, status TEXT);
        CREATE TABLE order_items (order_id TEXT REFERENCES orders, sku TEXT REFERENCES products, quantity INTEGER);
        CREATE TABLE tickets (ticket_id TEXT PRIMARY KEY, order_id TEXT REFERENCES orders, customer_id INTEGER REFERENCES customers,
                              created TEXT, category TEXT, priority TEXT, status TEXT);
    """)
    con.executemany("INSERT INTO products VALUES (?, ?, ?)", [(sku, name, price) for sku, (name, price) in CATALOG.items()])
    con.executemany("INSERT INTO customers VALUES (?, ?, ?, ?)", CUSTOMERS)
    con.executemany("INSERT INTO orders VALUES (?, ?, ?, ?)", ORDERS)
    con.executemany("INSERT INTO order_items VALUES (?, ?, ?)", ORDER_ITEMS)
    con.executemany("INSERT INTO tickets VALUES (?, ?, ?, ?, ?, ?, ?)", TICKET_ROWS)


def run_query(sql, params=()):
    """Runs a query on a READ-ONLY connection: the model's SQL can read the shop data, but never change it."""
    with sqlite3.connect("file:shop.db?mode=ro", uri=True) as con:
        return con.execute(sql, params).fetchall()


print("shop.db is ready. Orders in the database:", run_query("SELECT COUNT(*) FROM orders")[0][0])

# XML for the ERP System

The shop's old ERP system only reads XML, in this format. As for JSON in 6.01, the format goes into the system prompt.

There is no total in the XML: the ERP system computes it from the items, because arithmetic is a job for code, not for an LLM [Gao et al. (2023)](https://arxiv.org/abs/2211.10435).

In [ ]:
XML_FORMAT = """<ticket id="T-1999" priority="medium">
  <customer>Anna Schmidt</customer>
  <order id="ORD-10006"/>
  <items>
    <item sku="WC-600" quantity="1" unit_price="79.00"/>
  </items>
  <action>replacement</action>
</ticket>"""

XML_SYSTEM = f"""You turn support e-mails of an online shop into XML for the ERP system. Answer with the XML only.

Format (an example):
{XML_FORMAT}

Rules:
- priority: high (customer cannot work, money charged wrongly, deadline within 2 days), medium (product broken or missing,
  but the customer can still work), low (everything else)
- items: only the products the ticket is about. unit_price is the price from the catalog.
- action: refund, replacement, information or invoice_correction

Catalog (SKU: name, price in EUR):
""" + "\n".join(f"{sku}: {name}, {price:.2f}" for sku, (name, price) in CATALOG.items())

Let's try it on one e-mail, with a few seeds:

In [ ]:
xml_messages = [
    ("system", XML_SYSTEM),
    ("human", "Ticket id: T-2004\n\n" + TICKETS["T-2004"]),
]
print("### Ticket:")
print(TICKETS["T-2004"])
print("### LLM generated XMLs:")
for seed in [42, 1, 4321]:
    xml_answer = llm.invoke(xml_messages, seed=seed).content
    print(f"===== seed {seed}")
    print(xml_answer)

`XMLOutputParser` turns XML into Python dictionaries, as `PydanticOutputParser` does for JSON:

In [ ]:
XMLOutputParser().parse(xml_answer)

Let's look at the outcome: where are the attributes `sku`, `quantity` and `unit_price`? `XMLOutputParser` only keeps elements and their text; it drops attributes. The ERP system needs them, so we check the XML with Python's `ElementTree` ourselves: it reads both: an attribute with `item.get("sku")`, the text of an element with `root.findtext("customer")`.

**Attribute or element?** Both are valid XML; there is no fixed rule. The rule of thumb:

| | Attribute | Element |
|---|---|---|
| for | a short, single value about the element: `id`, `sku`, `unit` | the content itself, or what can be long, repeat, or have structure |
| example | `<order id="ORD-10004"/>` | `<customer>Mehmet Yilmaz</customer>` |
| cannot | repeat, or contain other elements | |

In practice, both are mixed, for example in e-invoices (UBL): `<InvoicedQuantity unitCode="EA">1</InvoicedQuantity>`.

## The Code Verifier

The code verifier checks the same three levels as in 6.01. For the ticket XML, these are the rules:

| Level | Rules |
|---|---|
| syntax | valid XML, the root element is `<ticket>` |
| schema | `<ticket>` has an `id` and a `priority` (`high`, `medium`, `low`); `<customer>` is not empty; `<order>` has an `id`; `<action>` is one of the four actions; every `<item>` has a whole-number `quantity` > 0 |
| business rules | the `<order>` exists in `shop.db` and belongs to this customer; every `<item>` is in the catalog and in the order, not more often than ordered; `unit_price` = catalog price |

The code is ready. `verify_ticket_xml()` returns `(ok, error_message)`, like `verify_joke` in 6.01.

In practice, an XML schema is written as an **XSD** file and checked by a library, for example `lxml`. We write the checks by hand, so you see every rule.

In [ ]:
def verify_xml_syntax(answer):
    try:
        root = ET.fromstring(answer.strip())
    except ET.ParseError as error:
        return False, f"invalid XML: {error}"
    if root.tag != "ticket":
        return False, f"the root element must be <ticket>, not <{root.tag}>"
    return True, ""



ACTIONS = ["refund", "replacement", "information", "invoice_correction"]


def check_ticket_xml(answer):
    """Checks the business rules. Returns (ok, rule, facts): the rule that failed, and what is wrong, with the correct values."""
    ok, error = verify_xml_syntax(answer)
    if not ok:
        return False, "The answer must be valid XML with the root element <ticket>, and nothing else.", error
    root = ET.fromstring(answer.strip())

    # 1. id and priority
    if not root.get("id"):
        return False, "<ticket> needs an id attribute.", "Use the ticket id from the user message."
    if root.get("priority") not in ["high", "medium", "low"]:
        return False, "priority must be high, medium or low.", f"Your priority is {root.get('priority')!r}."

    # 2. customer and action
    customer = (root.findtext("customer") or "").strip()
    if not customer:
        return False, "<customer> must contain the name of the customer.", "Your <customer> is missing or empty."
    action = (root.findtext("action") or "").strip()
    if action not in ACTIONS:
        return False, "<action> must be refund, replacement, information or invoice_correction.", f"Your action is {action!r}."

    # 3. the order is there, exists, and belongs to the customer
    order = root.find("order")
    if order is None or not order.get("id"):
        orders = run_query("SELECT o.order_id FROM orders o JOIN customers c USING (customer_id) WHERE c.name = ?", (customer,))
        return False, '<ticket> needs <order id="..."/> with the order number from the e-mail.', f"Your XML has no order id. The orders of {customer} are: {', '.join(o for (o,) in orders) or 'none'}."
    order_id = order.get("id", "")
    owner = run_query("SELECT c.name FROM orders o JOIN customers c USING (customer_id) WHERE o.order_id = ?", (order_id,))
    if not owner or owner[0][0] != customer:
        orders = run_query("SELECT o.order_id FROM orders o JOIN customers c USING (customer_id) WHERE c.name = ?", (customer,))
        facts = f"Order {order_id!r} does not exist." if not owner else f"Order {order_id} belongs to {owner[0][0]}."
        return False, "<order> must be an order of this customer.", f"{facts} The orders of {customer} are: {', '.join(o for (o,) in orders) or 'none'}."
    ordered = dict(run_query("SELECT sku, quantity FROM order_items WHERE order_id = ?", (order_id,)))  # sku -> quantity

    # 4. the items
    items = root.findall("items/item")
    if not items:
        return False, "<items> needs at least one <item>.", "Your <items> is missing or empty."
    for item in items:
        sku = item.get("sku")
        if sku not in CATALOG:
            return False, "Every sku must be a SKU from the catalog.", f"{sku!r} is not in the catalog."
        quantity = item.get("quantity") or ""
        if not quantity.isdigit() or int(quantity) < 1:
            return False, "quantity must be a whole number greater than 0.", f"Your quantity of {sku} is {quantity!r}."
        quantity = int(quantity)
        if sku not in ordered:
            return False, "Every item must be in the order.", f"{sku} is not in order {order_id}."
        if quantity > ordered[sku]:
            return False, "An item cannot be in the ticket more often than in the order.", f"Your quantity of {sku} is {quantity}, but order {order_id} has only {ordered[sku]}."
        try:
            unit_price = float(item.get("unit_price"))
        except (TypeError, ValueError):
            unit_price = None
        if unit_price is None or abs(unit_price - CATALOG[sku][1]) > 0.01:
            return False, "unit_price must be the price from the catalog.", f"Your unit_price of {sku} is {item.get('unit_price')!r}; the catalog price is {CATALOG[sku][1]:.2f}."


    return True, "", ""


def verify_ticket_xml(answer):
    """The code verifier: returns (ok, error_message), like verify_joke in 6.01."""
    ok, rule, facts = check_ticket_xml(answer)
    return (True, "") if ok else (False, f"{rule} {facts}")

Three wrong answers, one per level:

In [ ]:
wrong_answer_1 = """<ticket id="T-2004" priority="medium">
  <customer>Mehmet Yilmaz</customer>
  <order id="ORD-10004"/>
  <items>
    <item sku="LP-500" quantity="1" unit_price="39.90"/>
  <action>replacement</action>
</ticket>"""  # syntax: </items> is missing

wrong_answer_2 = """<ticket id="T-2004" priority="urgent">
  <customer>Mehmet Yilmaz</customer>
  <order id="ORD-10004"/>
  <items>
    <item sku="LP-500" quantity="1" unit_price="39.90"/>
  </items>
  <action>replacement</action>
</ticket>"""  # schema: "urgent" is not a priority

wrong_answer_3 = """<ticket id="T-2004" priority="medium">
  <customer>Mehmet Yilmaz</customer>
  <order id="ORD-10004"/>
  <items>
    <item sku="LP-500" quantity="1" unit_price="39.00"/>
  </items>
  <action>replacement</action>
</ticket>"""  # business rule: the laptop stand costs 39.90

print(f"{'what is wrong in the answer':30} | {'valid?':6} | the error message that the model gets")
print("-" * 130)
for wrong, wrong_answer in [("syntax: </items> is missing", wrong_answer_1), ("schema: priority 'urgent'", wrong_answer_2),
                            ("business rule: wrong price", wrong_answer_3)]:
    ok, error = verify_ticket_xml(wrong_answer)
    print(f"{wrong:30} | {'yes' if ok else 'no':6} | {error}")

## The Retry

The retry is the same `generate_with_verifier()` as in 6.01. Only the verifier is new: `verify_ticket_xml()`. If it finds an error, the model gets its XML and the error back, and tries again.

In [ ]:
from langchain_core.messages import convert_to_messages

RETRY_MESSAGE = "Your answer is not valid:\n{error}\nAnswer again and fix the problem."


def generate_with_verifier(generate, verify, messages, max_tries=3, feedback=True, verbose=False):
    """The retry. Returns (answer, number of tries, error); the error is None if the answer passed the verifier.
    feedback=False: every try gets the same prompt again (a new sample, but no error message)."""
    history = list(messages)
    for attempt in range(1, max_tries + 1):
        answer = generate(history)
        ok, error = verify(answer)
        if verbose:
            print(f"--- try {attempt}: {'ok' if ok else 'not valid'}\n{answer}\n" + ("" if ok else f"\nThe error:\n{error}\n"))
        if ok:
            return answer, attempt, None
        if feedback:
            history += [("ai", answer), ("human", RETRY_MESSAGE.format(error=error))]
    return answer, max_tries, error

We print once what the model sees in the second try:

In [ ]:
def generate_xml(history):
    answer = llm.invoke(history, seed=42).content  # a seed whose first XML breaks a rule
    if len(history) == 4:  # the second try: print once what the model sees, and its answer
        for message in convert_to_messages(history + [("ai", answer)]):
            message.pretty_print()
    return answer


answer, tries, error = generate_with_verifier(generate_xml, verify_ticket_xml, xml_messages)
print(f"\nResult: the XML is {'valid' if error is None else 'still not valid (' + error + ')'} after {tries} {'try' if tries == 1 else 'tries'}.")

# Your Turn: A New Rule for the Verifier

The shop has a new business rule: **a replacement is only possible for a delivered order.** A parcel that is still on the way cannot be replaced.

1. Add the rule to `verify_with_new_rule()`. You need the action from the XML (`root.findtext("action")`), the order id (`root.find("order").get("id")`), and the status of the order from `shop.db` (`run_query()`).
2. Write the error message for the model: the rule, and the facts (which order, which status).
3. Run the two tests, then all eight tickets with the retry.

In [ ]:
def verify_with_new_rule(answer):
    """verify_ticket_xml() plus the new rule: a replacement needs a delivered order. Returns (ok, error_message)."""
    ok, error = verify_ticket_xml(answer)
    if not ok:
        return False, error
    root = ET.fromstring(answer.strip())
    # TODO: the new rule. Return (False, your error message) if the XML breaks it.

    return True, ""


on_the_way = """<ticket id="T-2007" priority="high">
  <customer>Clara Wagner</customer>
  <order id="ORD-10008"/>
  <items>
    <item sku="DK-400" quantity="2" unit_price="159.00"/>
  </items>
  <action>replacement</action>
</ticket>"""  # order ORD-10008 is still on the way (status: shipped)

delivered = """<ticket id="T-2001" priority="high">
  <customer>Anna Schmidt</customer>
  <order id="ORD-10001"/>
  <items>
    <item sku="DK-400" quantity="1" unit_price="159.00"/>
  </items>
  <action>replacement</action>
</ticket>"""  # order ORD-10001 is delivered

print(f"{'test case':36} | {'should be':9} | {'your verifier says':18} | the error message for the model")
print("-" * 130)
for case, xml, should_be in [("replacement, order still on the way", on_the_way, "not valid"), ("replacement, order delivered", delivered, "valid")]:
    ok, error = verify_with_new_rule(xml)
    print(f"{case:36} | {should_be:9} | {'valid' if ok else 'not valid':18} | {error or '(none)'}")

In [ ]:
print(f"{'ticket':7} | {'tries':5} | {'valid at the end?':17} | the last error, if the XML is still not valid")
print("-" * 110)
for ticket_id, text in TICKETS.items():
    messages = [("system", XML_SYSTEM), ("human", f"Ticket id: {ticket_id}\n\n{text}")]
    answer, tries, error = generate_with_verifier(lambda history: llm.invoke(history, seed=42).content, verify_with_new_rule, messages)
    print(f"{ticket_id:7} | {tries:5} | {'yes' if error is None else 'NO':17} | {'' if error is None else error[:150]}")

Let's look at the outcome: which tickets need more than one try, and which rule failed? Add `verbose=True` to `generate_with_verifier()` to see every try.

# What Code Cannot Check

Some answers pass every code rule and are still wrong. In T-2001, Anna wants a new docking station, but this XML asks for a refund:

In [ ]:
wrong_xml = """<ticket id="T-2001" priority="high">
  <customer>Anna Schmidt</customer>
  <order id="ORD-10001"/>
  <items>
    <item sku="DK-400" quantity="1" unit_price="159.00"/>
  </items>
  <action>refund</action>
</ticket>"""  # Anna wants a new docking station, not her money back

ok, error = verify_ticket_xml(wrong_xml)
print("The code verifier says:", "valid" if ok else f"not valid ({error})")

Let's look at the outcome: what does the code verifier say?

Code only checks the rules we wrote down, and `refund` is one of the four actions. To see this mistake, someone has to read the e-mail and compare it with the XML. In 6.03, an LLM does this: the **LLM judge**.

# What You Learned

* **XML:** as for JSON in 6.01, the format goes into the system prompt. `XMLOutputParser` drops the attributes, so we check with `ElementTree`.
* The **code verifier** checks syntax, schema and business rules (the database), and returns `(ok, error_message)`.
* The **retry** is the same as in 6.01: the answer and the error go back to the model.
* A new business rule is a few lines in the verifier: the check, and an error message with the facts for the model.
* A code verifier only checks the rules we wrote down. An XML can pass all of them and still not fit the e-mail: for that, 6.03 adds an LLM judge.